In [ ]:
import pandas as pd
from collections import Counter

# Загружаем файл
file_path = "selected_keywords-4.xlsx"
sheet_names = pd.ExcelFile(file_path).sheet_names

# Берём 2-ю, 3-ю и 4-ю вкладки (индексы 1, 3, 4)
selected_sheets = [sheet_names[1], sheet_names[2], sheet_names[3]]
print(f"Обрабатываем вкладки: {selected_sheets}")

# Загружаем данные с этих вкладок
sheets_data = {}
for sheet in selected_sheets:
    sheets_data[sheet] = pd.read_excel(file_path, sheet_name=sheet)

# Проверяем структуру
for sheet, df in sheets_data.items():
    print(f"{sheet}: {df.shape} — колонки: {list(df.columns[:6])}")

Обрабатываем вкладки: ['Анастасия Владимировна', 'Анна', 'Александра']
Анастасия Владимировна: (500, 10) — колонки: ['id', 'source', 'keyword', 'context', 'cognitive_type', 'subtype']
Анна: (500, 10) — колонки: ['id', 'source', 'keyword', 'context', 'cognitive_type', 'subtype']
Александра: (500, 10) — колонки: ['id', 'source', 'keyword', 'context', 'cognitive_type', 'subtype']


In [ ]:
# Функция для агрегации и поиска согласия
def find_consensus(sheets_data, column_name):
    """
    Возвращает DataFrame с уникальными (id, keyword, context),
    у которых значение в column_name совпадает хотя бы у 2 аннотаторов
    """
    records = []

    for sheet_name, df in sheets_data.items():
        for _, row in df.iterrows():
            key = (row['id'], row['keyword'], row['context'])
            value = row[column_name]
            records.append({
                'id': row['id'],
                'keyword': row['keyword'],
                'context': row['context'],
                'annotator': sheet_name,
                column_name: value
            })

    # Группируем по (id, keyword, context)
    grouped = {}
    for rec in records:
        key = (rec['id'], rec['keyword'], rec['context'])
        if key not in grouped:
            grouped[key] = []
        grouped[key].append(rec[column_name])

    # Отбираем те, где хотя бы 2 одинаковых значения
    consensus_items = []
    for key, values in grouped.items():
        # Считаем частоту каждого значения
        value_counts = Counter(values)
        # Если есть значение с частотой >= 2
        most_common_value, count = value_counts.most_common(1)[0]
        if count >= 2:
            consensus_items.append({
                'id': key[0],
                'keyword': key[1],
                'context': key[2],
                column_name: most_common_value,  # согласованное значение
                'agreement_count': count
            })

    return pd.DataFrame(consensus_items)

# Таблица 1: согласие по cognitive_type
df_cognitive = find_consensus(sheets_data, 'cognitive_type')
print(f"\nТаблица 1 (cognitive_type): {len(df_cognitive)} уникальных строк с согласием хотя бы 2 аннотаторов")
df_cognitive.head(10)


Таблица 1 (cognitive_type): 500 уникальных строк с согласием хотя бы 2 аннотаторов


,id,keyword,context,cognitive_type,agreement_count
0,id_752,животными,с людьми и животными,perceptual,3
1,id_283,сумку,она держит пакет или сумку,perceptual,3
2,id_374,красном,мужчина в красном,perceptual,3
3,id_76,века,конца XVIII–начала XIX века,interpretative,3
4,id_439,скалистый,скалистый берег,perceptual,3
5,id_52,цветок,в волосах белый цветок,perceptual,3
6,id_694,положение,высокий статус и положение,interpretative,3
7,id_703,деревья,несколько деревьев,perceptual,3
8,id_629,вдалеке,строения вдалеке,perceptual,3
9,id_862,желтом,в желтом платье,perceptual,3


In [ ]:
# Таблица 2: согласие по subtype
df_subtype = find_consensus(sheets_data, 'subtype')
print(f"\nТаблица 2 (subtype): {len(df_subtype)} уникальных строк с согласием хотя бы 2 аннотаторов")
df_subtype.head(10)


Таблица 2 (subtype): 452 уникальных строк с согласием хотя бы 2 аннотаторов


,id,keyword,context,subtype,agreement_count
0,id_752,животными,с людьми и животными,animal,3
1,id_283,сумку,она держит пакет или сумку,item,2
2,id_374,красном,мужчина в красном,color,3
3,id_76,века,конца XVIII–начала XIX века,time,3
4,id_439,скалистый,скалистый берег,nature and landscape,2
5,id_694,положение,высокий статус и положение,status marker,2
6,id_703,деревья,несколько деревьев,nature and landscape,3
7,id_629,вдалеке,строения вдалеке,spatial,2
8,id_862,желтом,в желтом платье,color,3
9,id_400,помещения,темнота помещения,spatial,2


In [ ]:
# Сохраняем результаты в Excel
output_path = "consensus_results.xlsx"
with pd.ExcelWriter(output_path) as writer:
    df_cognitive.to_excel(writer, sheet_name='cognitive_type_consensus', index=False)
    df_subtype.to_excel(writer, sheet_name='subtype_consensus', index=False)

print(f"\nРезультаты сохранены в {output_path}")


Результаты сохранены в consensus_results.xlsx


In [ ]:
# Дополнительно: статистика по вкладкам и несогласованным случаям
def consensus_stats(sheets_data, column_name):
    """Показывает статистику согласованности"""
    all_values = {}
    for sheet_name, df in sheets_data.items():
        for _, row in df.iterrows():
            key = (row['id'], row['keyword'], row['context'])
            if key not in all_values:
                all_values[key] = {}
            all_values[key][sheet_name] = row[column_name]

    # Статистика
    total = len(all_values)
    full_agreement = 0
    two_way_agreement = 0
    no_agreement = 0

    for key, ann_values in all_values.items():
        values_list = list(ann_values.values())
        if len(set(values_list)) == 1:
            full_agreement += 1
        elif any(Counter(values_list)[v] >= 2 for v in set(values_list)):
            two_way_agreement += 1
        else:
            no_agreement += 1

    print(f"\nСтатистика для {column_name}:")
    print(f"  Всего уникальных (id+keyword+context): {total}")
    print(f"  Полное согласие (3/3): {full_agreement}")
    print(f"  Согласие 2/3: {two_way_agreement}")
    print(f"  Нет согласия: {no_agreement}")

consensus_stats(sheets_data, 'cognitive_type')
consensus_stats(sheets_data, 'subtype')


Статистика для cognitive_type:
  Всего уникальных (id+keyword+context): 500
  Полное согласие (3/3): 405
  Согласие 2/3: 95
  Нет согласия: 0

Статистика для subtype:
  Всего уникальных (id+keyword+context): 500
  Полное согласие (3/3): 283
  Согласие 2/3: 169
  Нет согласия: 48


In [ ]:
from statsmodels.stats.inter_rater import fleiss_kappa
import numpy as np
import pandas as pd

def prepare_fleiss_matrix(sheets_data, column_name):
    """
    Подготавливает матрицу для расчёта каппы Флейса
    """
    print(f"\n--- Начинаем подготовку матрицы для {column_name} ---")

    # Собираем все уникальные значения
    all_categories = set()
    for df in sheets_data.values():
        all_categories.update(df[column_name].dropna().unique())
    categories = sorted(list(all_categories))
    cat_to_idx = {cat: i for i, cat in enumerate(categories)}
    print(f"Найдено категорий: {len(categories)}")
    print(f"Категории: {categories}")

    # Группируем по (id, keyword, context)
    grouped_annotations = {}
    for sheet_name, df in sheets_data.items():
        print(f"Обрабатываю вкладку '{sheet_name}', строк: {len(df)}")
        for _, row in df.iterrows():
            key = (row['id'], row['keyword'], row['context'])
            value = row[column_name]
            if pd.notna(value):
                if key not in grouped_annotations:
                    grouped_annotations[key] = []
                grouped_annotations[key].append((sheet_name, value))

    print(f"Всего уникальных элементов: {len(grouped_annotations)}")

    # Проверяем количество аннотаторов для каждого элемента
    n_raters_per_item = [len(ann) for ann in grouped_annotations.values()]
    if n_raters_per_item:
        print(f"Количество аннотаторов на элемент: мин={min(n_raters_per_item)}, "
              f"макс={max(n_raters_per_item)}, среднее={np.mean(n_raters_per_item):.2f}")
        print(f"Распределение: {dict(zip(*np.unique(n_raters_per_item, return_counts=True)))}")
    else:
        print("Нет данных для анализа!")
        return np.array([]), []

    # Находим максимальное количество аннотаторов
    max_raters = max(n_raters_per_item)
    print(f"Максимальное количество аннотаторов: {max_raters}")

    # Оставляем только элементы с максимальным количеством аннотаторов
    filtered_items = {}
    for key, annotations in grouped_annotations.items():
        if len(annotations) == max_raters:
            filtered_items[key] = annotations

    print(f"Оставлено элементов с {max_raters} аннотаторами: {len(filtered_items)} "
          f"(из {len(grouped_annotations)} всего)")

    if len(filtered_items) == 0:
        print("ОШИБКА: Нет элементов с одинаковым количеством аннотаторов!")
        return np.array([]), []

    # Строим матрицу
    n_items = len(filtered_items)
    n_cats = len(categories)
    matrix = np.zeros((n_items, n_cats), dtype=int)

    for i, (key, annotations) in enumerate(filtered_items.items()):
        for _, value in annotations:
            if value in cat_to_idx:
                matrix[i, cat_to_idx[value]] += 1

    print(f"Матрица создана: {matrix.shape[0]} строк, {matrix.shape[1]} столбцов")
    print(f"Сумма по строкам (должна быть {max_raters}): {matrix.sum(axis=1)[:5]}...")

    return matrix, categories

def calculate_kappa(sheets_data, column_name, label):
    """Рассчитывает и выводит каппу Флейса"""
    print(f"\n{'='*60}")
    print(f"РАСЧЕТ КАППЫ ДЛЯ {label.upper()}")
    print(f"{'='*60}")

    matrix, categories = prepare_fleiss_matrix(sheets_data, column_name)

    if matrix.size == 0 or matrix.shape[0] == 0:
        print(f"\n❌ Невозможно рассчитать каппу для {label} ({column_name})")
        print("Нет подходящих элементов для анализа")
        return None, None, None

    # Рассчитываем каппу
    try:
        kappa = fleiss_kappa(matrix, method='fleiss')

        # Интерпретация
        if kappa < 0:
            interpretation = "хуже случайного"
        elif kappa < 0.21:
            interpretation = "незначительное согласие"
        elif kappa < 0.41:
            interpretation = "удовлетворительное согласие"
        elif kappa < 0.61:
            interpretation = "умеренное согласие"
        elif kappa < 0.81:
            interpretation = "существенное согласие"
        else:
            interpretation = "почти полное согласие"

        print(f"\n{'='*50}")
        print(f"Каппа Флейса для {label} ({column_name})")
        print(f"{'='*50}")
        print(f"✅ Значение каппы: {kappa:.4f}")
        print(f"📊 Интерпретация: {interpretation}")
        print(f"📌 Количество элементов: {matrix.shape[0]}")
        print(f"🏷️ Количество категорий: {matrix.shape[1]}")
        print(f"📋 Категории: {categories[:10]}{'...' if len(categories)>10 else ''}")

        return kappa, matrix, categories

    except Exception as e:
        print(f"\n❌ Ошибка при расчете каппы: {e}")
        return None, None, None

# Запуск расчетов
print("="*60)
print("НАЧАЛО АНАЛИЗА")
print("="*60)

# Считаем для cognitive_type
kappa_cog, matrix_cog, cats_cog = calculate_kappa(sheets_data, 'cognitive_type', 'когнитивных типов')

# Считаем для subtype
kappa_sub, matrix_sub, cats_sub = calculate_kappa(sheets_data, 'subtype', 'подтипов')

print("\n" + "="*60)
print("АНАЛИЗ ЗАВЕРШЕН")
print("="*60)

НАЧАЛО АНАЛИЗА

РАСЧЕТ КАППЫ ДЛЯ КОГНИТИВНЫХ ТИПОВ

--- Начинаем подготовку матрицы для cognitive_type ---
Найдено категорий: 2
Категории: ['interpretative', 'perceptual']
Обрабатываю вкладку 'Анастасия Владимировна', строк: 500
Обрабатываю вкладку 'Анна', строк: 500
Обрабатываю вкладку 'Александра', строк: 500
Всего уникальных элементов: 500
Количество аннотаторов на элемент: мин=3, макс=3, среднее=3.00
Распределение: {np.int64(3): np.int64(500)}
Максимальное количество аннотаторов: 3
Оставлено элементов с 3 аннотаторами: 500 (из 500 всего)
Матрица создана: 500 строк, 2 столбцов
Сумма по строкам (должна быть 3): [3 3 3 3 3]...

Каппа Флейса для когнитивных типов (cognitive_type)
✅ Значение каппы: 0.7193
📊 Интерпретация: существенное согласие
📌 Количество элементов: 500
🏷️ Количество категорий: 2
📋 Категории: ['interpretative', 'perceptual']

РАСЧЕТ КАППЫ ДЛЯ ПОДТИПОВ

--- Начинаем подготовку матрицы для subtype ---
Найдено категорий: 22
Категории: ['action', 'activity', 'aesthetic eva

In [ ]:
# Загружаем согласованные ответы (gold standard) - НЕ ДЕДУПЛИЦИРУЕМ
consensus_cognitive = pd.read_excel('consensus_results.xlsx', sheet_name='cognitive_type_consensus')
consensus_subtype = pd.read_excel('consensus_results.xlsx', sheet_name='subtype_consensus')

# Только нормализуем названия
consensus_cognitive['cognitive_type'] = consensus_cognitive['cognitive_type'].replace('interpretative', 'interpretive')
consensus_subtype['subtype'] = consensus_subtype['subtype'].replace('interpretative', 'interpretive')

print(f"Gold standard cognitive_type: {len(consensus_cognitive)} (должно быть 100)")
print(f"Gold standard subtype: {len(consensus_subtype)} (должно быть 93)")

Gold standard cognitive_type: 500 (должно быть 100)
Gold standard subtype: 452 (должно быть 93)


# YANDEX

In [30]:
# Загружаем предсказания модели
model_predictions = pd.read_csv('annotations_df_yandex.csv')
print(f"Предсказаний модели ДО дедупликации: {len(model_predictions)}")

# Нормализуем
model_predictions['cognitive_type'] = model_predictions['cognitive_type'].replace('interpretative', 'interpretive')
model_predictions['subtype'] = model_predictions['subtype'].replace('interpretative', 'interpretive')

# ПРАВИЛЬНОЕ ИСПРАВЛЕНИЕ: дедуплицируем ТОЛЬКО model_predictions
# Если для одной комбинации (id, keyword, context) несколько предсказаний -
# оставляем первое (или можно агрегировать)
model_predictions = model_predictions.drop_duplicates(
    subset=['image_id', 'keyword', 'context'],
    keep='first'
)

print(f"Предсказаний модели ПОСЛЕ дедупликации: {len(model_predictions)}")

Предсказаний модели ДО дедупликации: 194306
Предсказаний модели ПОСЛЕ дедупликации: 189589


In [31]:
# Теперь фильтруем предсказания модели по ключам из gold standard
def filter_model_by_gold(model_df, gold_df, target_name):
    """Оставляет в model_df только те ключи, которые есть в gold_df"""

    # Создаём ключи
    gold_df['key'] = gold_df['id'].astype(str) + '|' + \
                     gold_df['keyword'].astype(str) + '|' + \
                     gold_df['context'].astype(str)

    model_df['key'] = model_df['image_id'].astype(str) + '|' + \
                      model_df['keyword'].astype(str) + '|' + \
                      model_df['context'].astype(str)

    # Множество ключей из gold standard
    gold_keys = set(gold_df['key'])

    # Фильтруем модель
    model_filtered = model_df[model_df['key'].isin(gold_keys)].copy()

    print(f"\n{target_name}:")
    print(f"  Ключей в gold standard: {len(gold_keys)}")
    print(f"  Ключей в model_df (после дедупликации): {len(model_df)}")
    print(f"  Совпало ключей: {len(model_filtered)}")

    # Проверяем, все ли ключи из gold нашлись
    found_keys = set(model_filtered['key'])
    missing_keys = gold_keys - found_keys

    if missing_keys:
        print(f"  ⚠️ {len(missing_keys)} ключей из gold standard НЕ НАЙДЕНЫ в model_df")
        # Показываем примеры отсутствующих
        missing_examples = gold_df[gold_df['key'].isin(missing_keys)][['id', 'keyword', 'context']].head(5)
        print(f"     Примеры отсутствующих ключей:")
        print(missing_examples.to_string(index=False))
    else:
        print(f"  ✅ Все {len(gold_keys)} ключей найдены в model_df")

    return model_filtered

# Фильтруем предсказания
model_cognitive_filtered = filter_model_by_gold(
    model_predictions,
    consensus_cognitive,
    'cognitive_type'
)

model_subtype_filtered = filter_model_by_gold(
    model_predictions,
    consensus_subtype,
    'subtype'
)


cognitive_type:
  Ключей в gold standard: 500
  Ключей в model_df (после дедупликации): 189589
  Совпало ключей: 500
  ✅ Все 500 ключей найдены в model_df

subtype:
  Ключей в gold standard: 452
  Ключей в model_df (после дедупликации): 189589
  Совпало ключей: 452
  ✅ Все 452 ключей найдены в model_df


In [32]:
# Подготавливаем данные для сравнения
def prepare_for_evaluation(gold_df, model_df, target_column):
    """Объединяет gold standard с предсказаниями модели"""

    # Создаём ключи
    gold_df['key'] = gold_df['id'].astype(str) + '|' + \
                     gold_df['keyword'].astype(str) + '|' + \
                     gold_df['context'].astype(str)

    model_df['key'] = model_df['image_id'].astype(str) + '|' + \
                      model_df['keyword'].astype(str) + '|' + \
                      model_df['context'].astype(str)

    # Объединяем
    merged = pd.merge(
        gold_df[['key', target_column]],
        model_df[['key', target_column]],
        on='key',
        suffixes=('_true', '_pred'),
        how='inner'
    )

    print(f"\n{target_column}:")
    print(f"  Объединено строк: {len(merged)}")
    print(f"  Ожидалось: {len(gold_df)}")

    if len(merged) != len(gold_df):
        print(f"  ⚠️ ПОТЕРЯНО: {len(gold_df) - len(merged)} строк")

    y_true = merged[f'{target_column}_true'].astype(str)
    y_pred = merged[f'{target_column}_pred'].astype(str)

    return y_true, y_pred, merged

# Подготавливаем финальные выборки
y_true_cog, y_pred_cog, merged_cog = prepare_for_evaluation(
    consensus_cognitive,
    model_cognitive_filtered,
    'cognitive_type'
)

y_true_sub, y_pred_sub, merged_sub = prepare_for_evaluation(
    consensus_subtype,
    model_subtype_filtered,
    'subtype'
)


cognitive_type:
  Объединено строк: 500
  Ожидалось: 500

subtype:
  Объединено строк: 452
  Ожидалось: 452


In [34]:
# Проверяем размеры
print("\n" + "="*60)
print("ФИНАЛЬНЫЕ РАЗМЕРЫ ВЫБОРОК")
print("="*60)
print(f"cognitive_type: {len(y_true_cog)} образцов (должно быть 100)")
print(f"subtype: {len(y_true_sub)} образцов (должно быть 93)")

if len(y_true_cog) == 500 and len(y_true_sub) == 452:
    print("\n✅ ИДЕАЛЬНО! Размеры выборок совпадают!")
else:
    print("\n⚠️ Размеры не совпадают. Проверяем причину...")

    # Проверяем, нет ли дубликатов в gold standard
    cog_dups = consensus_cognitive.duplicated(subset=['id', 'keyword', 'context']).sum()
    sub_dups = consensus_subtype.duplicated(subset=['id', 'keyword', 'context']).sum()

    print(f"\nДубликатов в gold standard cognitive_type: {cog_dups}")
    print(f"Дубликатов в gold standard subtype: {sub_dups}")

    if cog_dups > 0 or sub_dups > 0:
        print("⚠️ НАЙДЕНЫ ДУБЛИКАТЫ В GOLD STANDARD! Их нужно удалить.")
        # Удаляем дубликаты из gold standard
        consensus_cognitive = consensus_cognitive.drop_duplicates(subset=['id', 'keyword', 'context'], keep='first')
        consensus_subtype = consensus_subtype.drop_duplicates(subset=['id', 'keyword', 'context'], keep='first')
        print(f"После удаления дубликатов cognitive_type: {len(consensus_cognitive)}")
        print(f"После удаления дубликатов subtype: {len(consensus_subtype)}")


ФИНАЛЬНЫЕ РАЗМЕРЫ ВЫБОРОК
cognitive_type: 500 образцов (должно быть 100)
subtype: 452 образцов (должно быть 93)

✅ ИДЕАЛЬНО! Размеры выборок совпадают!


In [35]:
# Считаем метрики
from sklearn.metrics import f1_score, precision_score, recall_score, classification_report, accuracy_score

def calculate_metrics(y_true, y_pred, target_name):
    """Рассчитывает все метрики"""

    print(f"\n{'='*60}")
    print(f"МЕТРИКИ ДЛЯ {target_name}")
    print(f"{'='*60}")
    print(f"Размер выборки: {len(y_true)}")

    accuracy = accuracy_score(y_true, y_pred)
    f1_macro = f1_score(y_true, y_pred, average='macro', zero_division=0)
    f1_weighted = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    f1_micro = f1_score(y_true, y_pred, average='micro', zero_division=0)

    print(f"\nAccuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
    print(f"F1-Macro: {f1_macro:.4f}")
    print(f"F1-Weighted: {f1_weighted:.4f}")
    print(f"F1-Micro: {f1_micro:.4f}")

    print(f"\nClassification Report:")
    print(classification_report(y_true, y_pred, zero_division=0))

    return {
        'target': target_name,
        'n_samples': len(y_true),
        'accuracy': accuracy,
        'f1_macro': f1_macro,
        'f1_weighted': f1_weighted
    }

metrics_cog = calculate_metrics(y_true_cog, y_pred_cog, "COGNITIVE_TYPE")
metrics_sub = calculate_metrics(y_true_sub, y_pred_sub, "SUBTYPE")


МЕТРИКИ ДЛЯ COGNITIVE_TYPE
Размер выборки: 500

Accuracy: 0.7980 (79.80%)
F1-Macro: 0.3619
F1-Weighted: 0.7719
F1-Micro: 0.7980

Classification Report:
              precision    recall  f1-score   support

interpretive       1.00      0.41      0.58       167
  perceptive       0.00      0.00      0.00         0
  perceptual       0.77      0.99      0.87       333
     spatial       0.00      0.00      0.00         0

    accuracy                           0.80       500
   macro avg       0.44      0.35      0.36       500
weighted avg       0.85      0.80      0.77       500


МЕТРИКИ ДЛЯ SUBTYPE
Размер выборки: 452

Accuracy: 0.6261 (62.61%)
F1-Macro: 0.4945
F1-Weighted: 0.6179
F1-Micro: 0.6261

Classification Report:
                      precision    recall  f1-score   support

              action       0.23      0.88      0.37         8
            activity       0.25      0.67      0.36         3
aesthetic evaluation       1.00      0.14      0.25        14
              ani

# GIGACHAT

In [36]:
# Загружаем предсказания модели
model_predictions = pd.read_csv('annotations_df_gigachat.csv')
print(f"Предсказаний модели ДО дедупликации: {len(model_predictions)}")

# Нормализуем
model_predictions['cognitive_type'] = model_predictions['cognitive_type'].replace('interpretative', 'interpretive')
model_predictions['subtype'] = model_predictions['subtype'].replace('interpretative', 'interpretive')

# ПРАВИЛЬНОЕ ИСПРАВЛЕНИЕ: дедуплицируем ТОЛЬКО model_predictions
# Если для одной комбинации (id, keyword, context) несколько предсказаний -
# оставляем первое (или можно агрегировать)
model_predictions = model_predictions.drop_duplicates(
    subset=['image_id', 'keyword', 'context'],
    keep='first'
)

print(f"Предсказаний модели ПОСЛЕ дедупликации: {len(model_predictions)}")

Предсказаний модели ДО дедупликации: 194306
Предсказаний модели ПОСЛЕ дедупликации: 189589


In [37]:
# Теперь фильтруем предсказания модели по ключам из gold standard
def filter_model_by_gold(model_df, gold_df, target_name):
    """Оставляет в model_df только те ключи, которые есть в gold_df"""

    # Создаём ключи
    gold_df['key'] = gold_df['id'].astype(str) + '|' + \
                     gold_df['keyword'].astype(str) + '|' + \
                     gold_df['context'].astype(str)

    model_df['key'] = model_df['image_id'].astype(str) + '|' + \
                      model_df['keyword'].astype(str) + '|' + \
                      model_df['context'].astype(str)

    # Множество ключей из gold standard
    gold_keys = set(gold_df['key'])

    # Фильтруем модель
    model_filtered = model_df[model_df['key'].isin(gold_keys)].copy()

    print(f"\n{target_name}:")
    print(f"  Ключей в gold standard: {len(gold_keys)}")
    print(f"  Ключей в model_df (после дедупликации): {len(model_df)}")
    print(f"  Совпало ключей: {len(model_filtered)}")

    # Проверяем, все ли ключи из gold нашлись
    found_keys = set(model_filtered['key'])
    missing_keys = gold_keys - found_keys

    if missing_keys:
        print(f"  ⚠️ {len(missing_keys)} ключей из gold standard НЕ НАЙДЕНЫ в model_df")
        # Показываем примеры отсутствующих
        missing_examples = gold_df[gold_df['key'].isin(missing_keys)][['id', 'keyword', 'context']].head(5)
        print(f"     Примеры отсутствующих ключей:")
        print(missing_examples.to_string(index=False))
    else:
        print(f"  ✅ Все {len(gold_keys)} ключей найдены в model_df")

    return model_filtered

# Фильтруем предсказания
model_cognitive_filtered = filter_model_by_gold(
    model_predictions,
    consensus_cognitive,
    'cognitive_type'
)

model_subtype_filtered = filter_model_by_gold(
    model_predictions,
    consensus_subtype,
    'subtype'
)


cognitive_type:
  Ключей в gold standard: 500
  Ключей в model_df (после дедупликации): 189589
  Совпало ключей: 500
  ✅ Все 500 ключей найдены в model_df

subtype:
  Ключей в gold standard: 452
  Ключей в model_df (после дедупликации): 189589
  Совпало ключей: 452
  ✅ Все 452 ключей найдены в model_df


In [38]:
# Подготавливаем данные для сравнения
def prepare_for_evaluation(gold_df, model_df, target_column):
    """Объединяет gold standard с предсказаниями модели"""

    # Создаём ключи
    gold_df['key'] = gold_df['id'].astype(str) + '|' + \
                     gold_df['keyword'].astype(str) + '|' + \
                     gold_df['context'].astype(str)

    model_df['key'] = model_df['image_id'].astype(str) + '|' + \
                      model_df['keyword'].astype(str) + '|' + \
                      model_df['context'].astype(str)

    # Объединяем
    merged = pd.merge(
        gold_df[['key', target_column]],
        model_df[['key', target_column]],
        on='key',
        suffixes=('_true', '_pred'),
        how='inner'
    )

    print(f"\n{target_column}:")
    print(f"  Объединено строк: {len(merged)}")
    print(f"  Ожидалось: {len(gold_df)}")

    if len(merged) != len(gold_df):
        print(f"  ⚠️ ПОТЕРЯНО: {len(gold_df) - len(merged)} строк")

    y_true = merged[f'{target_column}_true'].astype(str)
    y_pred = merged[f'{target_column}_pred'].astype(str)

    return y_true, y_pred, merged

# Подготавливаем финальные выборки
y_true_cog, y_pred_cog, merged_cog = prepare_for_evaluation(
    consensus_cognitive,
    model_cognitive_filtered,
    'cognitive_type'
)

y_true_sub, y_pred_sub, merged_sub = prepare_for_evaluation(
    consensus_subtype,
    model_subtype_filtered,
    'subtype'
)


cognitive_type:
  Объединено строк: 500
  Ожидалось: 500

subtype:
  Объединено строк: 452
  Ожидалось: 452


In [39]:
# Проверяем размеры
print("\n" + "="*60)
print("ФИНАЛЬНЫЕ РАЗМЕРЫ ВЫБОРОК")
print("="*60)
print(f"cognitive_type: {len(y_true_cog)} образцов (должно быть 100)")
print(f"subtype: {len(y_true_sub)} образцов (должно быть 93)")

if len(y_true_cog) == 500 and len(y_true_sub) == 452:
    print("\n✅ ИДЕАЛЬНО! Размеры выборок совпадают!")
else:
    print("\n⚠️ Размеры не совпадают. Проверяем причину...")

    # Проверяем, нет ли дубликатов в gold standard
    cog_dups = consensus_cognitive.duplicated(subset=['id', 'keyword', 'context']).sum()
    sub_dups = consensus_subtype.duplicated(subset=['id', 'keyword', 'context']).sum()

    print(f"\nДубликатов в gold standard cognitive_type: {cog_dups}")
    print(f"Дубликатов в gold standard subtype: {sub_dups}")

    if cog_dups > 0 or sub_dups > 0:
        print("⚠️ НАЙДЕНЫ ДУБЛИКАТЫ В GOLD STANDARD! Их нужно удалить.")
        # Удаляем дубликаты из gold standard
        consensus_cognitive = consensus_cognitive.drop_duplicates(subset=['id', 'keyword', 'context'], keep='first')
        consensus_subtype = consensus_subtype.drop_duplicates(subset=['id', 'keyword', 'context'], keep='first')
        print(f"После удаления дубликатов cognitive_type: {len(consensus_cognitive)}")
        print(f"После удаления дубликатов subtype: {len(consensus_subtype)}")


ФИНАЛЬНЫЕ РАЗМЕРЫ ВЫБОРОК
cognitive_type: 500 образцов (должно быть 100)
subtype: 452 образцов (должно быть 93)

✅ ИДЕАЛЬНО! Размеры выборок совпадают!


In [40]:
# Считаем метрики
from sklearn.metrics import f1_score, precision_score, recall_score, classification_report, accuracy_score

def calculate_metrics(y_true, y_pred, target_name):
    """Рассчитывает все метрики"""

    print(f"\n{'='*60}")
    print(f"МЕТРИКИ ДЛЯ {target_name}")
    print(f"{'='*60}")
    print(f"Размер выборки: {len(y_true)}")

    accuracy = accuracy_score(y_true, y_pred)
    f1_macro = f1_score(y_true, y_pred, average='macro', zero_division=0)
    f1_weighted = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    f1_micro = f1_score(y_true, y_pred, average='micro', zero_division=0)

    print(f"\nAccuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
    print(f"F1-Macro: {f1_macro:.4f}")
    print(f"F1-Weighted: {f1_weighted:.4f}")
    print(f"F1-Micro: {f1_micro:.4f}")

    print(f"\nClassification Report:")
    print(classification_report(y_true, y_pred, zero_division=0))

    return {
        'target': target_name,
        'n_samples': len(y_true),
        'accuracy': accuracy,
        'f1_macro': f1_macro,
        'f1_weighted': f1_weighted
    }

metrics_cog = calculate_metrics(y_true_cog, y_pred_cog, "COGNITIVE_TYPE")
metrics_sub = calculate_metrics(y_true_sub, y_pred_sub, "SUBTYPE")


МЕТРИКИ ДЛЯ COGNITIVE_TYPE
Размер выборки: 500

Accuracy: 0.8340 (83.40%)
F1-Macro: 0.8063
F1-Weighted: 0.8306
F1-Micro: 0.8340

Classification Report:
              precision    recall  f1-score   support

interpretive       0.79      0.68      0.73       167
  perceptual       0.85      0.91      0.88       333

    accuracy                           0.83       500
   macro avg       0.82      0.80      0.81       500
weighted avg       0.83      0.83      0.83       500


МЕТРИКИ ДЛЯ SUBTYPE
Размер выборки: 452

Accuracy: 0.5066 (50.66%)
F1-Macro: 0.4555
F1-Weighted: 0.5283
F1-Micro: 0.5066

Classification Report:
                      precision    recall  f1-score   support

              action       0.26      0.62      0.37         8
            activity       0.15      1.00      0.26         3
aesthetic evaluation       0.47      0.64      0.55        14
              animal       0.83      0.83      0.83        12
        architecture       0.71      0.83      0.77        18
 

# QWEN

In [41]:
# Загружаем предсказания модели
model_predictions = pd.read_csv('annotations_df_qwen.csv')
print(f"Предсказаний модели ДО дедупликации: {len(model_predictions)}")

# Нормализуем
model_predictions['cognitive_type'] = model_predictions['cognitive_type'].replace('interpretative', 'interpretive')
model_predictions['subtype'] = model_predictions['subtype'].replace('interpretative', 'interpretive')

# ПРАВИЛЬНОЕ ИСПРАВЛЕНИЕ: дедуплицируем ТОЛЬКО model_predictions
# Если для одной комбинации (id, keyword, context) несколько предсказаний -
# оставляем первое (или можно агрегировать)
model_predictions = model_predictions.drop_duplicates(
    subset=['image_id', 'keyword', 'context'],
    keep='first'
)

print(f"Предсказаний модели ПОСЛЕ дедупликации: {len(model_predictions)}")

Предсказаний модели ДО дедупликации: 194306
Предсказаний модели ПОСЛЕ дедупликации: 189589


In [42]:
# Теперь фильтруем предсказания модели по ключам из gold standard
def filter_model_by_gold(model_df, gold_df, target_name):
    """Оставляет в model_df только те ключи, которые есть в gold_df"""

    # Создаём ключи
    gold_df['key'] = gold_df['id'].astype(str) + '|' + \
                     gold_df['keyword'].astype(str) + '|' + \
                     gold_df['context'].astype(str)

    model_df['key'] = model_df['image_id'].astype(str) + '|' + \
                      model_df['keyword'].astype(str) + '|' + \
                      model_df['context'].astype(str)

    # Множество ключей из gold standard
    gold_keys = set(gold_df['key'])

    # Фильтруем модель
    model_filtered = model_df[model_df['key'].isin(gold_keys)].copy()

    print(f"\n{target_name}:")
    print(f"  Ключей в gold standard: {len(gold_keys)}")
    print(f"  Ключей в model_df (после дедупликации): {len(model_df)}")
    print(f"  Совпало ключей: {len(model_filtered)}")

    # Проверяем, все ли ключи из gold нашлись
    found_keys = set(model_filtered['key'])
    missing_keys = gold_keys - found_keys

    if missing_keys:
        print(f"  ⚠️ {len(missing_keys)} ключей из gold standard НЕ НАЙДЕНЫ в model_df")
        # Показываем примеры отсутствующих
        missing_examples = gold_df[gold_df['key'].isin(missing_keys)][['id', 'keyword', 'context']].head(5)
        print(f"     Примеры отсутствующих ключей:")
        print(missing_examples.to_string(index=False))
    else:
        print(f"  ✅ Все {len(gold_keys)} ключей найдены в model_df")

    return model_filtered

# Фильтруем предсказания
model_cognitive_filtered = filter_model_by_gold(
    model_predictions,
    consensus_cognitive,
    'cognitive_type'
)

model_subtype_filtered = filter_model_by_gold(
    model_predictions,
    consensus_subtype,
    'subtype'
)


cognitive_type:
  Ключей в gold standard: 500
  Ключей в model_df (после дедупликации): 189589
  Совпало ключей: 500
  ✅ Все 500 ключей найдены в model_df

subtype:
  Ключей в gold standard: 452
  Ключей в model_df (после дедупликации): 189589
  Совпало ключей: 452
  ✅ Все 452 ключей найдены в model_df


In [43]:
# Подготавливаем данные для сравнения
def prepare_for_evaluation(gold_df, model_df, target_column):
    """Объединяет gold standard с предсказаниями модели"""

    # Создаём ключи
    gold_df['key'] = gold_df['id'].astype(str) + '|' + \
                     gold_df['keyword'].astype(str) + '|' + \
                     gold_df['context'].astype(str)

    model_df['key'] = model_df['image_id'].astype(str) + '|' + \
                      model_df['keyword'].astype(str) + '|' + \
                      model_df['context'].astype(str)

    # Объединяем
    merged = pd.merge(
        gold_df[['key', target_column]],
        model_df[['key', target_column]],
        on='key',
        suffixes=('_true', '_pred'),
        how='inner'
    )

    print(f"\n{target_column}:")
    print(f"  Объединено строк: {len(merged)}")
    print(f"  Ожидалось: {len(gold_df)}")

    if len(merged) != len(gold_df):
        print(f"  ⚠️ ПОТЕРЯНО: {len(gold_df) - len(merged)} строк")

    y_true = merged[f'{target_column}_true'].astype(str)
    y_pred = merged[f'{target_column}_pred'].astype(str)

    return y_true, y_pred, merged

# Подготавливаем финальные выборки
y_true_cog, y_pred_cog, merged_cog = prepare_for_evaluation(
    consensus_cognitive,
    model_cognitive_filtered,
    'cognitive_type'
)

y_true_sub, y_pred_sub, merged_sub = prepare_for_evaluation(
    consensus_subtype,
    model_subtype_filtered,
    'subtype'
)


cognitive_type:
  Объединено строк: 500
  Ожидалось: 500

subtype:
  Объединено строк: 452
  Ожидалось: 452


In [44]:
# Проверяем размеры
print("\n" + "="*60)
print("ФИНАЛЬНЫЕ РАЗМЕРЫ ВЫБОРОК")
print("="*60)
print(f"cognitive_type: {len(y_true_cog)} образцов (должно быть 100)")
print(f"subtype: {len(y_true_sub)} образцов (должно быть 93)")

if len(y_true_cog) == 500 and len(y_true_sub) == 452:
    print("\n✅ ИДЕАЛЬНО! Размеры выборок совпадают!")
else:
    print("\n⚠️ Размеры не совпадают. Проверяем причину...")

    # Проверяем, нет ли дубликатов в gold standard
    cog_dups = consensus_cognitive.duplicated(subset=['id', 'keyword', 'context']).sum()
    sub_dups = consensus_subtype.duplicated(subset=['id', 'keyword', 'context']).sum()

    print(f"\nДубликатов в gold standard cognitive_type: {cog_dups}")
    print(f"Дубликатов в gold standard subtype: {sub_dups}")

    if cog_dups > 0 or sub_dups > 0:
        print("⚠️ НАЙДЕНЫ ДУБЛИКАТЫ В GOLD STANDARD! Их нужно удалить.")
        # Удаляем дубликаты из gold standard
        consensus_cognitive = consensus_cognitive.drop_duplicates(subset=['id', 'keyword', 'context'], keep='first')
        consensus_subtype = consensus_subtype.drop_duplicates(subset=['id', 'keyword', 'context'], keep='first')
        print(f"После удаления дубликатов cognitive_type: {len(consensus_cognitive)}")
        print(f"После удаления дубликатов subtype: {len(consensus_subtype)}")


ФИНАЛЬНЫЕ РАЗМЕРЫ ВЫБОРОК
cognitive_type: 500 образцов (должно быть 100)
subtype: 452 образцов (должно быть 93)

✅ ИДЕАЛЬНО! Размеры выборок совпадают!


In [45]:
# Считаем метрики
from sklearn.metrics import f1_score, precision_score, recall_score, classification_report, accuracy_score

def calculate_metrics(y_true, y_pred, target_name):
    """Рассчитывает все метрики"""

    print(f"\n{'='*60}")
    print(f"МЕТРИКИ ДЛЯ {target_name}")
    print(f"{'='*60}")
    print(f"Размер выборки: {len(y_true)}")

    accuracy = accuracy_score(y_true, y_pred)
    f1_macro = f1_score(y_true, y_pred, average='macro', zero_division=0)
    f1_weighted = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    f1_micro = f1_score(y_true, y_pred, average='micro', zero_division=0)

    print(f"\nAccuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
    print(f"F1-Macro: {f1_macro:.4f}")
    print(f"F1-Weighted: {f1_weighted:.4f}")
    print(f"F1-Micro: {f1_micro:.4f}")

    print(f"\nClassification Report:")
    print(classification_report(y_true, y_pred, zero_division=0))

    return {
        'target': target_name,
        'n_samples': len(y_true),
        'accuracy': accuracy,
        'f1_macro': f1_macro,
        'f1_weighted': f1_weighted
    }

metrics_cog = calculate_metrics(y_true_cog, y_pred_cog, "COGNITIVE_TYPE")
metrics_sub = calculate_metrics(y_true_sub, y_pred_sub, "SUBTYPE")


МЕТРИКИ ДЛЯ COGNITIVE_TYPE
Размер выборки: 500

Accuracy: 0.8760 (87.60%)
F1-Macro: 0.8456
F1-Weighted: 0.8683
F1-Micro: 0.8760

Classification Report:
              precision    recall  f1-score   support

interpretive       0.97      0.65      0.78       167
  perceptual       0.85      0.99      0.91       333

    accuracy                           0.88       500
   macro avg       0.91      0.82      0.85       500
weighted avg       0.89      0.88      0.87       500


МЕТРИКИ ДЛЯ SUBTYPE
Размер выборки: 452

Accuracy: 0.7257 (72.57%)
F1-Macro: 0.6738
F1-Weighted: 0.7250
F1-Micro: 0.7257

Classification Report:
                      precision    recall  f1-score   support

              action       0.41      0.88      0.56         8
            activity       0.43      1.00      0.60         3
aesthetic evaluation       0.67      0.57      0.62        14
              animal       1.00      0.75      0.86        12
        architecture       0.87      0.72      0.79        18
 

# MISTRAL

In [46]:
# Загружаем предсказания модели
model_predictions = pd.read_csv('annotations_df_mistral.csv')
print(f"Предсказаний модели ДО дедупликации: {len(model_predictions)}")

# Нормализуем
model_predictions['cognitive_type'] = model_predictions['cognitive_type'].replace('interpretative', 'interpretive')
model_predictions['subtype'] = model_predictions['subtype'].replace('interpretative', 'interpretive')

# ПРАВИЛЬНОЕ ИСПРАВЛЕНИЕ: дедуплицируем ТОЛЬКО model_predictions
# Если для одной комбинации (id, keyword, context) несколько предсказаний -
# оставляем первое (или можно агрегировать)
model_predictions = model_predictions.drop_duplicates(
    subset=['image_id', 'keyword', 'context'],
    keep='first'
)

print(f"Предсказаний модели ПОСЛЕ дедупликации: {len(model_predictions)}")

Предсказаний модели ДО дедупликации: 194306
Предсказаний модели ПОСЛЕ дедупликации: 189589


In [47]:
# Теперь фильтруем предсказания модели по ключам из gold standard
def filter_model_by_gold(model_df, gold_df, target_name):
    """Оставляет в model_df только те ключи, которые есть в gold_df"""

    # Создаём ключи
    gold_df['key'] = gold_df['id'].astype(str) + '|' + \
                     gold_df['keyword'].astype(str) + '|' + \
                     gold_df['context'].astype(str)

    model_df['key'] = model_df['image_id'].astype(str) + '|' + \
                      model_df['keyword'].astype(str) + '|' + \
                      model_df['context'].astype(str)

    # Множество ключей из gold standard
    gold_keys = set(gold_df['key'])

    # Фильтруем модель
    model_filtered = model_df[model_df['key'].isin(gold_keys)].copy()

    print(f"\n{target_name}:")
    print(f"  Ключей в gold standard: {len(gold_keys)}")
    print(f"  Ключей в model_df (после дедупликации): {len(model_df)}")
    print(f"  Совпало ключей: {len(model_filtered)}")

    # Проверяем, все ли ключи из gold нашлись
    found_keys = set(model_filtered['key'])
    missing_keys = gold_keys - found_keys

    if missing_keys:
        print(f"  ⚠️ {len(missing_keys)} ключей из gold standard НЕ НАЙДЕНЫ в model_df")
        # Показываем примеры отсутствующих
        missing_examples = gold_df[gold_df['key'].isin(missing_keys)][['id', 'keyword', 'context']].head(5)
        print(f"     Примеры отсутствующих ключей:")
        print(missing_examples.to_string(index=False))
    else:
        print(f"  ✅ Все {len(gold_keys)} ключей найдены в model_df")

    return model_filtered

# Фильтруем предсказания
model_cognitive_filtered = filter_model_by_gold(
    model_predictions,
    consensus_cognitive,
    'cognitive_type'
)

model_subtype_filtered = filter_model_by_gold(
    model_predictions,
    consensus_subtype,
    'subtype'
)


cognitive_type:
  Ключей в gold standard: 500
  Ключей в model_df (после дедупликации): 189589
  Совпало ключей: 500
  ✅ Все 500 ключей найдены в model_df

subtype:
  Ключей в gold standard: 452
  Ключей в model_df (после дедупликации): 189589
  Совпало ключей: 452
  ✅ Все 452 ключей найдены в model_df


In [48]:
# Подготавливаем данные для сравнения
def prepare_for_evaluation(gold_df, model_df, target_column):
    """Объединяет gold standard с предсказаниями модели"""

    # Создаём ключи
    gold_df['key'] = gold_df['id'].astype(str) + '|' + \
                     gold_df['keyword'].astype(str) + '|' + \
                     gold_df['context'].astype(str)

    model_df['key'] = model_df['image_id'].astype(str) + '|' + \
                      model_df['keyword'].astype(str) + '|' + \
                      model_df['context'].astype(str)

    # Объединяем
    merged = pd.merge(
        gold_df[['key', target_column]],
        model_df[['key', target_column]],
        on='key',
        suffixes=('_true', '_pred'),
        how='inner'
    )

    print(f"\n{target_column}:")
    print(f"  Объединено строк: {len(merged)}")
    print(f"  Ожидалось: {len(gold_df)}")

    if len(merged) != len(gold_df):
        print(f"  ⚠️ ПОТЕРЯНО: {len(gold_df) - len(merged)} строк")

    y_true = merged[f'{target_column}_true'].astype(str)
    y_pred = merged[f'{target_column}_pred'].astype(str)

    return y_true, y_pred, merged

# Подготавливаем финальные выборки
y_true_cog, y_pred_cog, merged_cog = prepare_for_evaluation(
    consensus_cognitive,
    model_cognitive_filtered,
    'cognitive_type'
)

y_true_sub, y_pred_sub, merged_sub = prepare_for_evaluation(
    consensus_subtype,
    model_subtype_filtered,
    'subtype'
)


cognitive_type:
  Объединено строк: 500
  Ожидалось: 500

subtype:
  Объединено строк: 452
  Ожидалось: 452


In [49]:
# Проверяем размеры
print("\n" + "="*60)
print("ФИНАЛЬНЫЕ РАЗМЕРЫ ВЫБОРОК")
print("="*60)
print(f"cognitive_type: {len(y_true_cog)} образцов (должно быть 100)")
print(f"subtype: {len(y_true_sub)} образцов (должно быть 93)")

if len(y_true_cog) == 500 and len(y_true_sub) == 452:
    print("\n✅ ИДЕАЛЬНО! Размеры выборок совпадают!")
else:
    print("\n⚠️ Размеры не совпадают. Проверяем причину...")

    # Проверяем, нет ли дубликатов в gold standard
    cog_dups = consensus_cognitive.duplicated(subset=['id', 'keyword', 'context']).sum()
    sub_dups = consensus_subtype.duplicated(subset=['id', 'keyword', 'context']).sum()

    print(f"\nДубликатов в gold standard cognitive_type: {cog_dups}")
    print(f"Дубликатов в gold standard subtype: {sub_dups}")

    if cog_dups > 0 or sub_dups > 0:
        print("⚠️ НАЙДЕНЫ ДУБЛИКАТЫ В GOLD STANDARD! Их нужно удалить.")
        # Удаляем дубликаты из gold standard
        consensus_cognitive = consensus_cognitive.drop_duplicates(subset=['id', 'keyword', 'context'], keep='first')
        consensus_subtype = consensus_subtype.drop_duplicates(subset=['id', 'keyword', 'context'], keep='first')
        print(f"После удаления дубликатов cognitive_type: {len(consensus_cognitive)}")
        print(f"После удаления дубликатов subtype: {len(consensus_subtype)}")


ФИНАЛЬНЫЕ РАЗМЕРЫ ВЫБОРОК
cognitive_type: 500 образцов (должно быть 100)
subtype: 452 образцов (должно быть 93)

✅ ИДЕАЛЬНО! Размеры выборок совпадают!


In [50]:
# Считаем метрики
from sklearn.metrics import f1_score, precision_score, recall_score, classification_report, accuracy_score

def calculate_metrics(y_true, y_pred, target_name):
    """Рассчитывает все метрики"""

    print(f"\n{'='*60}")
    print(f"МЕТРИКИ ДЛЯ {target_name}")
    print(f"{'='*60}")
    print(f"Размер выборки: {len(y_true)}")

    accuracy = accuracy_score(y_true, y_pred)
    f1_macro = f1_score(y_true, y_pred, average='macro', zero_division=0)
    f1_weighted = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    f1_micro = f1_score(y_true, y_pred, average='micro', zero_division=0)

    print(f"\nAccuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
    print(f"F1-Macro: {f1_macro:.4f}")
    print(f"F1-Weighted: {f1_weighted:.4f}")
    print(f"F1-Micro: {f1_micro:.4f}")

    print(f"\nClassification Report:")
    print(classification_report(y_true, y_pred, zero_division=0))

    return {
        'target': target_name,
        'n_samples': len(y_true),
        'accuracy': accuracy,
        'f1_macro': f1_macro,
        'f1_weighted': f1_weighted
    }

metrics_cog = calculate_metrics(y_true_cog, y_pred_cog, "COGNITIVE_TYPE")
metrics_sub = calculate_metrics(y_true_sub, y_pred_sub, "SUBTYPE")


МЕТРИКИ ДЛЯ COGNITIVE_TYPE
Размер выборки: 500

Accuracy: 0.8860 (88.60%)
F1-Macro: 0.5739
F1-Weighted: 0.8810
F1-Micro: 0.8860

Classification Report:
              precision    recall  f1-score   support

interpretive       0.97      0.68      0.80       167
  perceptual       0.86      0.99      0.92       333
     unknown       0.00      0.00      0.00         0

    accuracy                           0.89       500
   macro avg       0.61      0.56      0.57       500
weighted avg       0.90      0.89      0.88       500


МЕТРИКИ ДЛЯ SUBTYPE
Размер выборки: 452

Accuracy: 0.7412 (74.12%)
F1-Macro: 0.6382
F1-Weighted: 0.7430
F1-Micro: 0.7412

Classification Report:
                      precision    recall  f1-score   support

              action       0.86      0.75      0.80         8
            activity       0.23      1.00      0.38         3
aesthetic evaluation       0.73      0.57      0.64        14
              animal       1.00      0.92      0.96        12
        a

# GPT-OSS

In [51]:
# Загружаем предсказания модели
model_predictions = pd.read_csv('annotations_df_gpt-oss.csv')
print(f"Предсказаний модели ДО дедупликации: {len(model_predictions)}")

# Нормализуем
model_predictions['cognitive_type'] = model_predictions['cognitive_type'].replace('interpretative', 'interpretive')
model_predictions['subtype'] = model_predictions['subtype'].replace('interpretative', 'interpretive')

# ПРАВИЛЬНОЕ ИСПРАВЛЕНИЕ: дедуплицируем ТОЛЬКО model_predictions
# Если для одной комбинации (id, keyword, context) несколько предсказаний -
# оставляем первое (или можно агрегировать)
model_predictions = model_predictions.drop_duplicates(
    subset=['image_id', 'keyword', 'context'],
    keep='first'
)

print(f"Предсказаний модели ПОСЛЕ дедупликации: {len(model_predictions)}")

Предсказаний модели ДО дедупликации: 193724
Предсказаний модели ПОСЛЕ дедупликации: 189039


In [52]:
# Теперь фильтруем предсказания модели по ключам из gold standard
def filter_model_by_gold(model_df, gold_df, target_name):
    """Оставляет в model_df только те ключи, которые есть в gold_df"""

    # Создаём ключи
    gold_df['key'] = gold_df['id'].astype(str) + '|' + \
                     gold_df['keyword'].astype(str) + '|' + \
                     gold_df['context'].astype(str)

    model_df['key'] = model_df['image_id'].astype(str) + '|' + \
                      model_df['keyword'].astype(str) + '|' + \
                      model_df['context'].astype(str)

    # Множество ключей из gold standard
    gold_keys = set(gold_df['key'])

    # Фильтруем модель
    model_filtered = model_df[model_df['key'].isin(gold_keys)].copy()

    print(f"\n{target_name}:")
    print(f"  Ключей в gold standard: {len(gold_keys)}")
    print(f"  Ключей в model_df (после дедупликации): {len(model_df)}")
    print(f"  Совпало ключей: {len(model_filtered)}")

    # Проверяем, все ли ключи из gold нашлись
    found_keys = set(model_filtered['key'])
    missing_keys = gold_keys - found_keys

    if missing_keys:
        print(f"  ⚠️ {len(missing_keys)} ключей из gold standard НЕ НАЙДЕНЫ в model_df")
        # Показываем примеры отсутствующих
        missing_examples = gold_df[gold_df['key'].isin(missing_keys)][['id', 'keyword', 'context']].head(5)
        print(f"     Примеры отсутствующих ключей:")
        print(missing_examples.to_string(index=False))
    else:
        print(f"  ✅ Все {len(gold_keys)} ключей найдены в model_df")

    return model_filtered

# Фильтруем предсказания
model_cognitive_filtered = filter_model_by_gold(
    model_predictions,
    consensus_cognitive,
    'cognitive_type'
)

model_subtype_filtered = filter_model_by_gold(
    model_predictions,
    consensus_subtype,
    'subtype'
)


cognitive_type:
  Ключей в gold standard: 500
  Ключей в model_df (после дедупликации): 189039
  Совпало ключей: 499
  ⚠️ 1 ключей из gold standard НЕ НАЙДЕНЫ в model_df
     Примеры отсутствующих ключей:
    id      keyword                   context
id_722 благоговение благоговение и восхищение

subtype:
  Ключей в gold standard: 452
  Ключей в model_df (после дедупликации): 189039
  Совпало ключей: 451
  ⚠️ 1 ключей из gold standard НЕ НАЙДЕНЫ в model_df
     Примеры отсутствующих ключей:
    id      keyword                   context
id_722 благоговение благоговение и восхищение


In [53]:
# Подготавливаем данные для сравнения
def prepare_for_evaluation(gold_df, model_df, target_column):
    """Объединяет gold standard с предсказаниями модели"""

    # Создаём ключи
    gold_df['key'] = gold_df['id'].astype(str) + '|' + \
                     gold_df['keyword'].astype(str) + '|' + \
                     gold_df['context'].astype(str)

    model_df['key'] = model_df['image_id'].astype(str) + '|' + \
                      model_df['keyword'].astype(str) + '|' + \
                      model_df['context'].astype(str)

    # Объединяем
    merged = pd.merge(
        gold_df[['key', target_column]],
        model_df[['key', target_column]],
        on='key',
        suffixes=('_true', '_pred'),
        how='inner'
    )

    print(f"\n{target_column}:")
    print(f"  Объединено строк: {len(merged)}")
    print(f"  Ожидалось: {len(gold_df)}")

    if len(merged) != len(gold_df):
        print(f"  ⚠️ ПОТЕРЯНО: {len(gold_df) - len(merged)} строк")

    y_true = merged[f'{target_column}_true'].astype(str)
    y_pred = merged[f'{target_column}_pred'].astype(str)

    return y_true, y_pred, merged

# Подготавливаем финальные выборки
y_true_cog, y_pred_cog, merged_cog = prepare_for_evaluation(
    consensus_cognitive,
    model_cognitive_filtered,
    'cognitive_type'
)

y_true_sub, y_pred_sub, merged_sub = prepare_for_evaluation(
    consensus_subtype,
    model_subtype_filtered,
    'subtype'
)


cognitive_type:
  Объединено строк: 499
  Ожидалось: 500
  ⚠️ ПОТЕРЯНО: 1 строк

subtype:
  Объединено строк: 451
  Ожидалось: 452
  ⚠️ ПОТЕРЯНО: 1 строк


In [54]:
# Проверяем размеры
print("\n" + "="*60)
print("ФИНАЛЬНЫЕ РАЗМЕРЫ ВЫБОРОК")
print("="*60)
print(f"cognitive_type: {len(y_true_cog)} образцов (должно быть 100)")
print(f"subtype: {len(y_true_sub)} образцов (должно быть 93)")

if len(y_true_cog) == 499 and len(y_true_sub) == 451:
    print("\n✅ ИДЕАЛЬНО! Размеры выборок совпадают!")
else:
    print("\n⚠️ Размеры не совпадают. Проверяем причину...")

    # Проверяем, нет ли дубликатов в gold standard
    cog_dups = consensus_cognitive.duplicated(subset=['id', 'keyword', 'context']).sum()
    sub_dups = consensus_subtype.duplicated(subset=['id', 'keyword', 'context']).sum()

    print(f"\nДубликатов в gold standard cognitive_type: {cog_dups}")
    print(f"Дубликатов в gold standard subtype: {sub_dups}")

    if cog_dups > 0 or sub_dups > 0:
        print("⚠️ НАЙДЕНЫ ДУБЛИКАТЫ В GOLD STANDARD! Их нужно удалить.")
        # Удаляем дубликаты из gold standard
        consensus_cognitive = consensus_cognitive.drop_duplicates(subset=['id', 'keyword', 'context'], keep='first')
        consensus_subtype = consensus_subtype.drop_duplicates(subset=['id', 'keyword', 'context'], keep='first')
        print(f"После удаления дубликатов cognitive_type: {len(consensus_cognitive)}")
        print(f"После удаления дубликатов subtype: {len(consensus_subtype)}")


ФИНАЛЬНЫЕ РАЗМЕРЫ ВЫБОРОК
cognitive_type: 499 образцов (должно быть 100)
subtype: 451 образцов (должно быть 93)

✅ ИДЕАЛЬНО! Размеры выборок совпадают!


In [55]:
# Считаем метрики
from sklearn.metrics import f1_score, precision_score, recall_score, classification_report, accuracy_score

def calculate_metrics(y_true, y_pred, target_name):
    """Рассчитывает все метрики"""

    print(f"\n{'='*60}")
    print(f"МЕТРИКИ ДЛЯ {target_name}")
    print(f"{'='*60}")
    print(f"Размер выборки: {len(y_true)}")

    accuracy = accuracy_score(y_true, y_pred)
    f1_macro = f1_score(y_true, y_pred, average='macro', zero_division=0)
    f1_weighted = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    f1_micro = f1_score(y_true, y_pred, average='micro', zero_division=0)

    print(f"\nAccuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
    print(f"F1-Macro: {f1_macro:.4f}")
    print(f"F1-Weighted: {f1_weighted:.4f}")
    print(f"F1-Micro: {f1_micro:.4f}")

    print(f"\nClassification Report:")
    print(classification_report(y_true, y_pred, zero_division=0))

    return {
        'target': target_name,
        'n_samples': len(y_true),
        'accuracy': accuracy,
        'f1_macro': f1_macro,
        'f1_weighted': f1_weighted
    }

metrics_cog = calculate_metrics(y_true_cog, y_pred_cog, "COGNITIVE_TYPE")
metrics_sub = calculate_metrics(y_true_sub, y_pred_sub, "SUBTYPE")


МЕТРИКИ ДЛЯ COGNITIVE_TYPE
Размер выборки: 499

Accuracy: 0.8537 (85.37%)
F1-Macro: 0.8184
F1-Weighted: 0.8452
F1-Micro: 0.8537

Classification Report:
              precision    recall  f1-score   support

interpretive       0.91      0.62      0.74       166
  perceptual       0.84      0.97      0.90       333

    accuracy                           0.85       499
   macro avg       0.87      0.80      0.82       499
weighted avg       0.86      0.85      0.85       499


МЕТРИКИ ДЛЯ SUBTYPE
Размер выборки: 451

Accuracy: 0.6940 (69.40%)
F1-Macro: 0.5712
F1-Weighted: 0.7111
F1-Micro: 0.6940

Classification Report:
                      precision    recall  f1-score   support

              action       0.75      0.75      0.75         8
            activity       0.14      0.33      0.20         3
aesthetic evaluation       0.88      0.50      0.64        14
              animal       1.00      0.83      0.91        12
        architecture       0.90      0.50      0.64        18
 